In [1]:
# Fine-Tuning TinyLlama 1.1B Chat on GSM8K (NF4 QLoRA)
### Base Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
### Quantization: 4-bit NF4 (NormalFloat4) with double quantization — the QLoRA paper's exact setup
### Fine-tuning: LoRA adapters via `peft`, trained with TRL's `SFTTrainer`
### Dataset: GSM8K (openai/gsm8k, "main" config) — official train / test split
### Goal: Improve step-by-step math word problem solving, then export a servable model for a FastAPI endpoint

'''**Why NF4 specifically:** NF4 is a quantization data type designed for normally-distributed weights
(which is what pretrained LLM weights look like). At the same 4-bit budget, NF4 preserves more
information than plain FP4/INT4, which is why it's the default in the original QLoRA paper. Combined
with double quantization (quantizing the quantization constants themselves), it roughly halves memory
vs standard 4-bit while keeping accuracy close to full 16-bit fine-tuning.

**Pipeline in this notebook:**
1. Install deps
2. Load base model in NF4 (explicit `BitsAndBytesConfig`)
3. Attach LoRA adapters with `peft`
4. Load GSM8K train + test splits
5. Format data into Llama-3 chat template
6. Baseline zero-shot evaluation (before fine-tuning) on test set
7. Fine-tune with SFTTrainer
8. Save LoRA adapter
9. Merge LoRA into base weights (in fp16) and save a full model for serving
10. Evaluate fine-tuned model, compare accuracy to baseline'''

"**Why NF4 specifically:** NF4 is a quantization data type designed for normally-distributed weights\n(which is what pretrained LLM weights look like). At the same 4-bit budget, NF4 preserves more\ninformation than plain FP4/INT4, which is why it's the default in the original QLoRA paper. Combined\nwith double quantization (quantizing the quantization constants themselves), it roughly halves memory\nvs standard 4-bit while keeping accuracy close to full 16-bit fine-tuning.\n\n**Pipeline in this notebook:**\n1. Install deps\n2. Load base model in NF4 (explicit `BitsAndBytesConfig`)\n3. Attach LoRA adapters with `peft`\n4. Load GSM8K train + test splits\n5. Format data into Llama-3 chat template\n6. Baseline zero-shot evaluation (before fine-tuning) on test set\n7. Fine-tune with SFTTrainer\n8. Save LoRA adapter\n9. Merge LoRA into base weights (in fp16) and save a full model for serving\n10. Evaluate fine-tuned model, compare accuracy to baseline"

In [2]:
!pip install -q -U transformers==4.46.3 accelerate==1.1.1 peft==0.13.2 \
    bitsandbytes trl==0.12.0 datasets huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 81.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 44.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 83.0 MB/s eta 0:00:00:00:01


In [4]:
### 2. Load base model with explicit NF4 quantization

'''This is the standard QLoRA config:
- `load_in_4bit=True` — store weights in 4-bit
- `bnb_4bit_quant_type="nf4"` — use the NF4 data type (vs default "fp4")
- `bnb_4bit_use_double_quant=True` — quantize the quantization constants too (extra memory savings)
- `bnb_4bit_compute_dtype=torch.bfloat16` — de-quantize to bf16 on the fly for matmuls (keeps compute accurate)'''
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_SEQ_LENGTH = 512

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
)
model.config.use_cache = False  # required for gradient checkpointing during training

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
### 3. Attach LoRA adapters with `peft`
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [6]:
### 4. Load GSM8K dataset (official train + test split)
from datasets import load_dataset

raw_train = load_dataset("openai/gsm8k", "main", split="train")
raw_test  = load_dataset("openai/gsm8k", "main", split="test")

print("Train size:", len(raw_train))
print("Test size :", len(raw_test))
print(raw_train[0])

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Train size: 7473
Test size : 1319
{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


In [7]:
'''### 5. Parse GSM8K format

Each GSM8K example looks like:
```
question: "Natalia sold clips to 48 of her friends..."
answer:   "Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72"
```
The `answer` field already contains step-by-step reasoning, ending in `#### <final_numeric_answer>`.
We'll:
- Strip the `<<...>>` calculator annotations (cleaner reasoning text)
- Extract the final numeric answer for use in evaluation later
'''
import re

def clean_reasoning(answer_text: str) -> str:
    """Remove GSM8K's <<calculator annotations>> to get clean natural reasoning."""
    return re.sub(r"<<[^>]*>>", "", answer_text)

def extract_final_answer(answer_text: str):
    """Pull the numeric value after the final '#### '."""
    match = re.search(r"####\s*([\-0-9,\.]+)", answer_text)
    if not match:
        return None
    return match.group(1).replace(",", "").strip()

# quick sanity check
sample = raw_train[0]
print(clean_reasoning(sample["answer"]))
print("Final answer ->", extract_final_answer(sample["answer"]))


Natalia sold 48/2 = 24 clips in May.
Natalia sold 48+24 = 72 clips altogether in April and May.
#### 72
Final answer -> 72


In [8]:
### 6. Format into Llama-3 chat template
SYSTEM_PROMPT = (
    "You are a careful math tutor. Solve the word problem step by step, "
    "showing your reasoning clearly, then give the final numeric answer "
    "on its own line in the form: #### <answer>."
)

def format_chat(example):
    reasoning = clean_reasoning(example["answer"])
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": reasoning},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_dataset = raw_train.map(format_chat)
print(train_dataset[0]["text"][:800])


Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

<|system|>
You are a careful math tutor. Solve the word problem step by step, showing your reasoning clearly, then give the final numeric answer on its own line in the form: #### <answer>.</s>
<|user|>
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?</s>
<|assistant|>
Natalia sold 48/2 = 24 clips in May.
Natalia sold 48+24 = 72 clips altogether in April and May.
#### 72</s>



In [9]:
###. Token budget check
def count_tokens(dataset, tokenizer, field="text"):
    return sum(len(tokenizer(example[field]).input_ids) for example in dataset)

total_tokens = count_tokens(train_dataset, tokenizer)
print(f"Total training tokens: {total_tokens:,}")
print(f"Avg tokens/example   : {total_tokens/len(train_dataset):.1f}")


Total training tokens: 1,743,817
Avg tokens/example   : 233.3


In [11]:
### Fine-tune with TRL SFTTrainer
from trl import SFTTrainer, SFTConfig

model.config.use_cache = False  # keep disabled during training (re-enable for inference later)

training_args = SFTConfig(
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    warmup_steps=20,
    num_train_epochs=3,
    learning_rate=2e-4,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=20,
    optim="paged_adamw_8bit",    # paged optimizer, standard pairing with QLoRA
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    output_dir="outputs",
    report_to="none",
    dataset_text_field="text",
    max_seq_length=512,
    packing=True,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
)


Generating train split: 0 examples [00:00, ? examples/s]

In [12]:
trainer_stats = trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,1.449200
40,1.014800
60,0.962600
80,0.930600
100,0.902500
120,0.887900
140,0.891500
160,0.873300
180,0.864900
200,0.855100


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


KeyboardInterrupt: 

In [13]:
### Save LoRA adapter
ADAPTER_DIR = "tinyllama-1.1b-gsm8k-nf4-lora-adapter"

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA adapter saved to ./{ADAPTER_DIR}")


LoRA adapter saved to ./tinyllama-1.1b-gsm8k-nf4-lora-adapter


In [14]:
### Merge LoRA into base weights and save the FULL model
from peft import PeftModel

MERGED_DIR = "tinyllama-1.1b-gsm8k-nf4-merged"

# Reload base model in bf16 (unquantized) for a numerically clean merge
base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

merged_model = PeftModel.from_pretrained(base_model_fp16, ADAPTER_DIR)
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to ./{MERGED_DIR}")


Merged model saved to ./tinyllama-1.1b-gsm8k-nf4-merged


In [34]:
# Upload to hugging Face
from huggingface_hub import HfApi, create_repo, notebook_login
notebook_login()

In [35]:
REPO_ID = "raturihimanshu077/tinyllama-1.1b-gsm8k-nf4"  # change to your username
create_repo(REPO_ID, private=True, exist_ok=True)

api = HfApi()
api.upload_folder(
    folder_path="tinyllama-1.1b-gsm8k-nf4-merged",
    repo_id=REPO_ID,
    repo_type="model",
)
print(f"Uploaded to https://huggingface.co/{REPO_ID}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded to https://huggingface.co/raturihimanshu077/tinyllama-1.1b-gsm8k-nf4
